## 1. Imports

In [1]:
%pip install numpy scipy

   ---------------------------------------- 0.0/37.3 MB ? eta -:--:--
   - -------------------------------------- 1.0/37.3 MB 5.4 MB/s eta 0:00:07
   - -------------------------------------- 1.3/37.3 MB 3.1 MB/s eta 0:00:12
   -- ------------------------------------- 2.4/37.3 MB 4.0 MB/s eta 0:00:09
   --- ------------------------------------ 3.4/37.3 MB 4.2 MB/s eta 0:00:09
   ----- ---------------------------------- 4.7/37.3 MB 4.6 MB/s eta 0:00:08
   ------ --------------------------------- 6.0/37.3 MB 4.9 MB/s eta 0:00:07
   ------- -------------------------------- 7.3/37.3 MB 5.1 MB/s eta 0:00:06
   -------- ------------------------------- 8.4/37.3 MB 5.1 MB/s eta 0:00:06
   ---------- ----------------------------- 10.0/37.3 MB 5.4 MB/s eta 0:00:06
   ------------ --------------------------- 11.5/37.3 MB 5.6 MB/s eta 0:00:05
   ------------- -------------------------- 12.3/37.3 MB 5.5 MB/s eta 0:00:05
   -------------- ------------------------- 13.6/37.3 MB 5.5 MB/s eta 0:00:05
  

In [2]:
import cv2
import numpy as np
from scipy.optimize import linear_sum_assignment

## 2. Setup

In [3]:
#VIDEO_PATH = r"C:\projects\motion-capture-project\data\v01.mp4"
VIDEO_PATH = r"C:\projects\retargeting-mocap-project\data\test-video-2.mp4"

MIN_AREA = 20
MAX_AREA = 2000
MAX_DISTANCE = 50

LOWER_WHITE = np.array([0, 0, 180])
UPPER_WHITE = np.array([180, 80, 255])

## 3. Function to detect markers

In [4]:
def detect_markers(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    mask = cv2.inRange(hsv, LOWER_WHITE, UPPER_WHITE)

    kernel = np.ones((5, 5), np.uint8)

    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask)

    detections = []

    for label in range(1, num_labels):
        x, y, w, h, area = stats[label]

        if area < MIN_AREA:
            continue

        if area > MAX_AREA:
            continue

        cx, cy = centroids[label]
        detections.append([cx, cy])

    return np.array(detections, dtype=np.float32), mask

## 4. Test detection on the first frame

In [5]:
cap = cv2.VideoCapture(VIDEO_PATH)

ok, frame = cap.read()
cap.release()

detections, mask = detect_markers(frame)

print("Detecciones:", len(detections))
print(detections[:10])

Detecciones: 21
[[565.41455 295.26068]
 [442.10855 298.5559 ]
 [421.06522 412.06522]
 [335.1295  421.1367 ]
 [613.9464  439.2902 ]
 [493.4425  544.9443 ]
 [251.39038 642.97864]
 [482.40237 662.8669 ]
 [684.17255 669.86725]
 [377.48264 790.70654]]


## 5. View detections

In [8]:
debug = frame.copy()

for x, y in detections:
    cv2.circle(debug, (int(x), int(y)), 6, (0, 0, 255), 2)

cv2.imwrite(r"C:\projects\retargeting-mocap-project\outputs\imgs\debug_detections_frame0.png", debug)
cv2.imwrite(r"C:\projects\retargeting-mocap-project\outputs\imgs\debug_mask_frame0.png", mask)

print("Saved: debug_detections_frame0.png")
print("Saved: debug_mask_frame0.png")

Saved: debug_detections_frame0.png
Saved: debug_mask_frame0.png


## 6. Create Kalman

In [9]:
def create_kalman(x, y):
    kf = cv2.KalmanFilter(4, 2)

    kf.transitionMatrix = np.array([
        [1, 0, 1, 0],
        [0, 1, 0, 1],
        [0, 0, 1, 0],
        [0, 0, 0, 1],
    ], dtype=np.float32)

    kf.measurementMatrix = np.array([
        [1, 0, 0, 0],
        [0, 1, 0, 0],
    ], dtype=np.float32)

    kf.processNoiseCov = np.eye(4, dtype=np.float32) * 0.03
    kf.measurementNoiseCov = np.eye(2, dtype=np.float32) * 1.0
    kf.errorCovPost = np.eye(4, dtype=np.float32)

    kf.statePost = np.array([[x], [y], [0], [0]], dtype=np.float32)

    return kf

## 7. Run full tracking run

In [10]:
MAX_MISSING_FRAMES = 30

cap = cv2.VideoCapture(VIDEO_PATH)

tracks = []
frame_idx = 0

while True:
    ok, frame = cap.read()
    if not ok:
        break

    detections, mask = detect_markers(frame)

    if frame_idx == 0:
        for x, y in detections:
            tracks.append({
                "kalman": create_kalman(x, y),
                "history": [[x, y]],
                "missing": 0,
                "active": True
            })

    else:
        predictions = []

        for track in tracks:
            pred = track["kalman"].predict()
            predictions.append([pred[0, 0], pred[1, 0]])

        predictions = np.array(predictions, dtype=np.float32)

        if len(detections) > 0 and len(predictions) > 0:
            cost = np.zeros((len(predictions), len(detections)), dtype=np.float32)

            for i, pred in enumerate(predictions):
                for j, det in enumerate(detections):
                    cost[i, j] = np.linalg.norm(pred - det)

            rows, cols = linear_sum_assignment(cost)

            assigned_tracks = set()
            assigned_detections = set()

            for r, c in zip(rows, cols):
                if tracks[r]["missing"] > MAX_MISSING_FRAMES:
                    continue

                if cost[r, c] > MAX_DISTANCE:
                    continue

                x, y = detections[c]

                measurement = np.array([[x], [y]], dtype=np.float32)
                tracks[r]["kalman"].correct(measurement)

                tracks[r]["history"].append([x, y])
                tracks[r]["missing"] = 0
                tracks[r]["active"] = True

                assigned_tracks.add(r)
                assigned_detections.add(c)

            for i, track in enumerate(tracks):
                if i not in assigned_tracks:
                    pred = predictions[i]

                    if track["missing"] <= MAX_MISSING_FRAMES:
                        #track["history"].append([pred[0], pred[1]])
                        track["history"].append([np.nan, np.nan])
                        track["missing"] += 1
                        track["active"] = False
                    else:
                        track["history"].append([np.nan, np.nan])

            for j, det in enumerate(detections):
                if j not in assigned_detections:
                    x, y = det

                    tracks.append({
                        "kalman": create_kalman(x, y),
                        "history": [[np.nan, np.nan]] * frame_idx + [[x, y]],
                        "missing": 0,
                        "active": True
                    })

        else:
            for track in tracks:
                pred = track["kalman"].predict()

                if track["missing"] <= MAX_MISSING_FRAMES:
                    #track["history"].append([pred[0, 0], pred[1, 0]])
                    track["history"].append([np.nan, np.nan])
                    track["missing"] += 1
                    track["active"] = False
                else:
                    track["history"].append([np.nan, np.nan])

    frame_idx += 1

cap.release()

print("Frames procesados:", frame_idx)
print("Tracks encontrados:", len(tracks))

Frames procesados: 389
Tracks encontrados: 172


In [11]:
cap = cv2.VideoCapture(VIDEO_PATH)

tracks = []
frame_idx = 0

while True:
    ok, frame = cap.read()
    if not ok:
        break

    detections, mask = detect_markers(frame)

    if frame_idx == 0:
        for x, y in detections:
            tracks.append({
                "kalman": create_kalman(x, y),
                "history": [[x, y]]
            })

    else:
        predictions = []

        for track in tracks:
            pred = track["kalman"].predict()
            predictions.append([pred[0, 0], pred[1, 0]])

        predictions = np.array(predictions, dtype=np.float32)

        if len(detections) > 0 and len(predictions) > 0:
            cost = np.zeros((len(predictions), len(detections)), dtype=np.float32)

            for i, pred in enumerate(predictions):
                for j, det in enumerate(detections):
                    cost[i, j] = np.linalg.norm(pred - det)

            rows, cols = linear_sum_assignment(cost)

            assigned_tracks = set()
            assigned_detections = set()

            for r, c in zip(rows, cols):
                if cost[r, c] > MAX_DISTANCE:
                    continue

                x, y = detections[c]

                measurement = np.array([[x], [y]], dtype=np.float32)
                tracks[r]["kalman"].correct(measurement)
                tracks[r]["history"].append([x, y])

                assigned_tracks.add(r)
                assigned_detections.add(c)

            for i, track in enumerate(tracks):
                if i not in assigned_tracks:
                    pred = predictions[i]
                    #track["history"].append([pred[0], pred[1]])
                    track["history"].append([np.nan, np.nan])

            for j, det in enumerate(detections):
                if j not in assigned_detections:
                    x, y = det
                    tracks.append({
                        "kalman": create_kalman(x, y),
                        "history": [[np.nan, np.nan]] * frame_idx + [[x, y]]
                    })

        else:
            for track in tracks:
                pred = track["kalman"].predict()
                #track["history"].append([pred[0, 0], pred[1, 0]])
                track["history"].append([np.nan, np.nan])

    frame_idx += 1

cap.release()

print("Frames processed:", frame_idx)
print("Tracks found:", len(tracks))

Frames processed: 389
Tracks found: 98


## 8. Convert to array

In [12]:
max_len = max(len(t["history"]) for t in tracks)

for t in tracks:
    while len(t["history"]) < max_len:
        t["history"].append([np.nan, np.nan])

trajectories = np.array([t["history"] for t in tracks], dtype=np.float32)
trajectories = np.transpose(trajectories, (1, 0, 2))

print(trajectories.shape)

(389, 98, 2)


## 9. Save trajectories

In [13]:
np.save(r"C:\projects\retargeting-mocap-project\outputs\others\trajectories_2d.npy", trajectories)

print("Guardado: trajectories_2d.npy")

Guardado: trajectories_2d.npy


## 10. Review a trajectory

In [14]:
frame_id = 0
marker_id = 0

print(trajectories[frame_id, marker_id])

[565.41455 295.26068]
